# Bulldozer Detection Model Testing
This notebook covers dataset loading, analysis, and model training for bulldozer detection using YOLOv8.

## 1. Setup and Imports

In [18]:
%pip install ultralytics
import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO
import glob

# Define Input Paths
BULLDOZER_PATH = '/kaggle/input/bulldozer-v1i-yolov8'
EXCAVATOR_PATH = '/kaggle/input/excavator' # Verify this name in your sidebar!
WORKING_DIR = '/kaggle/working/combined_dataset'

# Create folder structure for the combined dataset
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{WORKING_DIR}/{split}/images', exist_ok=True)
    os.makedirs(f'{WORKING_DIR}/{split}/labels', exist_ok=True)

Note: you may need to restart the kernel to use updated packages.


## 2. Dataset Loading and Configuration

In [ ]:
def merge_datasets(src_path, dst_path, class_offset=0):
    for split in ['train', 'valid', 'test']:
        img_src = os.path.join(src_path, split, 'images')
        lbl_src = os.path.join(src_path, split, 'labels')
        
        if not os.path.exists(img_src): continue
            
        # 1. Symlink Images (saves disk space)
        for img_file in os.listdir(img_src):
            src_file = os.path.join(img_src, img_file)
            dst_file = os.path.join(dst_path, split, 'images', f"{class_offset}_{img_file}")
            if not os.path.exists(dst_file):
                os.symlink(src_file, dst_file)
        
        # 2. Copy and Remap Labels
        for lbl_file in os.listdir(lbl_src):
            src_file = os.path.join(lbl_src, lbl_file)
            dst_file = os.path.join(dst_path, split, 'labels', f"{class_offset}_{lbl_file}")
            
            with open(src_file, 'r') as f:
                lines = f.readlines()
            
            with open(dst_file, 'w') as f:
                for line in lines:
                    parts = line.split()
                    # Increment class ID by the offset
                    parts[0] = str(int(parts[0]) + class_offset)
                    f.write(" ".join(parts) + "\n")

# Process both datasets
merge_datasets(BULLDOZER_PATH, WORKING_DIR, class_offset=0) # Bulldozer = Class 0
merge_datasets(EXCAVATOR_PATH, WORKING_DIR, class_offset=1) # Excavator = Class 1

# Create the new combined YAML
data_config = {
    'path': WORKING_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['bulldozer', 'excavator']
}

with open('/kaggle/working/combined_data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("Combined dataset created with 2 classes!")

In [ ]:
# Define paths using the Kaggle input directory
base_path = Path(KAGGLE_INPUT_DIR)

# Use glob to find images (supports both .jpg and .png)
train_images = list((base_path / 'train' / 'images').glob('*.*'))
valid_images = list((base_path / 'valid' / 'images').glob('*.*'))
test_images = list((base_path / 'test' / 'images').glob('*.*'))

# Filter to ensure we only have images
extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
train_images = [x for x in train_images if x.suffix.lower() in extensions]
valid_images = [x for x in valid_images if x.suffix.lower() in extensions]
test_images = [x for x in test_images if x.suffix.lower() in extensions]

print(f"Train images: {len(train_images)}")
print(f"Validation images: {len(valid_images)}")
print(f"Test images: {len(test_images)}")
print(f"Total images: {len(train_images) + len(valid_images) + len(test_images)}")

## 3. Dataset Analysis

### 3.1 Image Size Distribution

In [ ]:
def get_image_sizes(image_paths):
    """Get dimensions of all images"""
    sizes = []
    for img_path in image_paths:
        img = Image.open(img_path)
        sizes.append(img.size)  # (width, height)
    return sizes

train_sizes = get_image_sizes(train_images[:20])  # Sample first 20 for speed
widths = [s[0] for s in train_sizes]
heights = [s[1] for s in train_sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=10, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Image Width Distribution')

axes[1].hist(heights, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Image Height Distribution')

plt.tight_layout()
plt.show()

print(f"Average width: {np.mean(widths):.0f} pixels")
print(f"Average height: {np.mean(heights):.0f} pixels")

### 3.2 Annotation Analysis

In [ ]:
def count_annotations(image_paths):
    """Count objects in YOLO format annotations"""
    annotation_counts = []
    
    for img_path in image_paths:
        # Get corresponding label file
        label_path = str(img_path).replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
                annotation_counts.append(len(lines))
        else:
            annotation_counts.append(0)
    
    return annotation_counts

train_annotations = count_annotations(train_images)
valid_annotations = count_annotations(valid_images)
test_annotations = count_annotations(test_images)

print(f"Training set:")
print(f"  Total annotations: {sum(train_annotations)}")
print(f"  Average per image: {np.mean(train_annotations):.2f}")
print(f"  Max per image: {max(train_annotations)}")

print(f"\nValidation set:")
print(f"  Total annotations: {sum(valid_annotations)}")
print(f"  Average per image: {np.mean(valid_annotations):.2f}")

print(f"\nTest set:")
print(f"  Total annotations: {sum(test_annotations)}")
print(f"  Average per image: {np.mean(test_annotations):.2f}")

In [ ]:
# Visualize annotation distribution
plt.figure(figsize=(10, 5))
plt.hist(train_annotations, bins=range(max(train_annotations)+2), edgecolor='black', alpha=0.7)
plt.xlabel('Number of bulldozers per image')
plt.ylabel('Frequency')
plt.title('Annotation Distribution in Training Set')
plt.grid(alpha=0.3)
plt.show()

### 3.3 Sample Images Visualization

In [ ]:
def visualize_samples(image_paths, num_samples=6):
    """Visualize sample images with annotations"""
    samples = np.random.choice(image_paths, min(num_samples, len(image_paths)), replace=False)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, img_path in enumerate(samples):
        # Load image
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Load annotations
        label_path = str(img_path).replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, x_center, y_center, width, height = map(float, parts)
                        
                        # Convert YOLO format to pixel coordinates
                        x1 = int((x_center - width/2) * w)
                        y1 = int((y_center - height/2) * h)
                        x2 = int((x_center + width/2) * w)
                        y2 = int((y_center + height/2) * h)
                        
                        # Draw bounding box
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cv2.putText(img, 'bulldozer', (x1, y1-10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f'Image {idx+1}')
    
    plt.tight_layout()
    plt.show()

print("Training Set Samples:")
visualize_samples(train_images, num_samples=6)

## 4. Model Training

In [ ]:
model = YOLO('yolov8n.pt') 

results = model.train(
    data=working_yaml_path,  # Use the modified yaml in /kaggle/working/
    epochs=50,
    imgsz=416,
    project='/kaggle/working/runs', # Save results to working dir
    name='bulldozer_experiment'
)

### 4.2 Train the Model

### 4.3 View Training Results

In [ ]:
# Display training results
results_path = Path('/kaggle/working/runs/detect/bulldozer_detect')

# Check if results directory exists
if results_path.exists():
    # Display training curves
    results_img = results_path / 'results.png'
    if results_img.exists():
        img = Image.open(results_img)
        plt.figure(figsize=(16, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Training Results')
        plt.show()
    
    # Display confusion matrix
    confusion_matrix = results_path / 'confusion_matrix.png'
    if confusion_matrix.exists():
        img = Image.open(confusion_matrix)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Confusion Matrix')
        plt.show()
else:
    print("Training results not found. Please run training first.")

## 5. Model Validation

In [ ]:
# Validate the model on the validation set
metrics = model.val(data=KAGGLE_INPUT_DIR+'/data.yaml')

print(f"\nValidation Metrics:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

## 6. Model Testing

### 6.1 Test on Validation Images

In [ ]:
# Run predictions on validation images
def test_predictions(image_paths, num_samples=6, conf=0.25):
    """Run predictions and visualize results"""
    samples = np.random.choice(image_paths, min(num_samples, len(image_paths)), replace=False)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, img_path in enumerate(samples):
        # Run prediction
        results = model.predict(str(img_path), conf=conf, verbose=False)
        
        # Get annotated image
        annotated_img = results[0].plot()
        annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
        
        # Count detections
        num_detections = len(results[0].boxes)
        
        axes[idx].imshow(annotated_img)
        axes[idx].axis('off')
        axes[idx].set_title(f'Detections: {num_detections}')
    
    plt.tight_layout()
    plt.show()

print("Predictions on Validation Set:")
test_predictions(valid_images, num_samples=6, conf=0.25)

### 6.2 Test on Test Set

In [ ]:
print("Predictions on Test Set:")
test_predictions(test_images, num_samples=6, conf=0.25)

### 6.3 Confidence Threshold Analysis

In [ ]:
# Test different confidence thresholds
confidence_thresholds = [0.1, 0.25, 0.5, 0.75]

# Pick one sample image
sample_img = valid_images[0]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, conf in enumerate(confidence_thresholds):
    results = model.predict(str(sample_img), conf=conf, verbose=False)
    annotated_img = results[0].plot()
    annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    num_detections = len(results[0].boxes)
    
    axes[idx].imshow(annotated_img)
    axes[idx].axis('off')
    axes[idx].set_title(f'Confidence: {conf} | Detections: {num_detections}')

plt.tight_layout()
plt.show()

## 7. Export Model

In [ ]:
# Export model to different formats
# Uncomment the format you need

# ONNX format (for deployment)
# model.export(format='onnx')

# TensorRT format (for NVIDIA GPUs)
# model.export(format='engine')

# TensorFlow SavedModel
# model.export(format='saved_model')

# TensorFlow Lite
# model.export(format='tflite')

print("To export the model, uncomment the desired format above and run the cell.")

## 8. Model Performance Summary

In [ ]:
# Create a summary of model performance
print("="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"\nDataset:")
print(f"  Training images: {len(train_images)}")
print(f"  Validation images: {len(valid_images)}")
print(f"  Test images: {len(test_images)}")
print(f"  Classes: {data_config['names']}")

print(f"\nModel: YOLOv8n")
print(f"\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

print(f"\nModel saved to: runs/detect/bulldozer_detect/weights/best.pt")
print("="*60)